In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.config import get_broad_field, infer_broad_field_from_text
from src.embeddings import load_model
from src.re_ranker import evaluate_reranker

data_dir = repo_root / 'data' / 'raw'
resume_path = data_dir / 'Resume' / 'Resume.csv'
posting_path = data_dir / 'postings.csv'
resume_embeddings_path = repo_root / 'data' / 'resume_embeddings.npy'

resumes_df = pd.read_csv(resume_path, encoding='utf-8')
postings_df = pd.read_csv(posting_path, encoding='utf-8')
resume_embeddings = np.load(resume_embeddings_path)

resumes_df = resumes_df.copy()
resumes_df['category'] = resumes_df['Category'].fillna('other')
resumes_df['broad_field'] = resumes_df['category'].apply(get_broad_field)
resumes_df['id'] = resumes_df['ID'].astype(str)

resumes_df = resumes_df.iloc[: resume_embeddings.shape[0]].copy()

jds_df = postings_df[['job_id', 'title', 'description']].copy()
jds_df = jds_df.rename(columns={'job_id': 'id'})
jds_df['title'] = jds_df['title'].fillna('')
jds_df['description'] = jds_df['description'].fillna('')
jds_df['text'] = jds_df['title'] + ' ' + jds_df['description']
jds_df['category'] = jds_df['text'].apply(infer_broad_field_from_text)
jds_df['broad_field'] = jds_df['category'].apply(get_broad_field)

jd_limit = 200
jds_df = jds_df.head(jd_limit).copy()

print('Resume rows:', len(resumes_df))
print('JD rows:', len(jds_df))
print('Resume embedding shape:', resume_embeddings.shape)

model = load_model()
jd_texts = jds_df['title'].fillna('') + ' ' + jds_df['description'].fillna('')
jd_embeddings = model.encode(jd_texts.tolist(), batch_size=32, show_progress_bar=False, normalize_embeddings=True)
print('JD embedding shape:', jd_embeddings.shape)


In [ ]:
results = evaluate_reranker(
    resumes_df,
    jds_df,
    resume_embeddings,
    jd_embeddings,
    top_k=25,
    sample_negatives=5,
    n_splits=3,
)

print('Baseline nDCG@10:', round(results['baseline_ndcg@10'], 4))
print('Reranker nDCG@10:', round(results['reranker_ndcg@10'], 4))
print('Fold metrics:')
print(results['fold_metrics'])
print('Top feature importances:')
print(results['feature_importances'].head(10))
